# 06: Evaluation Protocol

Section 3.5's shared evaluation setup for comparing the five representations (`raw`, `classical`, `ft_transformer`, `subtab`, `scarf`) built in `04_feature_encoding.ipynb`/`05_embedding_paradigms.ipynb`.

## Setup

In [ ]:
%matplotlib inline

import os

# torch bundles its own OpenMP runtime (libiomp5md.dll) while
# numpy/pandas/scikit-learn pull in MKL's (libomp.dll/libiomp5md.dll) --
# loading both into one process aborts the kernel outright ("OMP: Error
# #15: Initializing ..., but found ... already initialized"), not a
# Python exception, so it shows up as a bare kernel crash with no
# traceback. Must be set before numpy/torch are imported anywhere below.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path

from IPython.display import display

import lime.lime_tabular
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config
from src.evaluation import eval_setup

sns.set_theme(style="whitegrid")

ARTIFACTS_DIR = REPO_ROOT / "data" / "artifacts" / eval_setup.ACTIVE_CANDIDATE
FINAL_DIR = config.FINAL_DIR

## 1. Load representations and candidate splits

The one cell every later cell in this notebook depends on. Loads all five representations (`eval_setup.load_representations`) and the candidate's own split files (`eval_setup.load_candidate_splits`), then asserts row alignment between them (`eval_setup.assert_row_alignment`) before anything downstream trusts positional alignment between a representation frame and `candidate_splits`. Left uncaught -- if alignment is wrong, this cell must fail loudly, not be swallowed and discovered later as a silently wrong score.

In [ ]:
representations = eval_setup.load_representations(ARTIFACTS_DIR)
candidate_splits = eval_setup.load_candidate_splits(FINAL_DIR)

eval_setup.assert_row_alignment(representations, candidate_splits)

for repr_name in eval_setup.REPR_NAMES:
    for split in eval_setup.SPLITS:
        shape = representations[repr_name][split].shape
        print(f"representations[{repr_name!r}][{split!r}]: {shape}")

for split in eval_setup.SPLITS:
    print(f"candidate_splits[{split!r}]: {candidate_splits[split].shape}")

## 2. Derive the genre label

`eval_setup.derive_genre_label` (first-listed genre; empty-list rows fall back to the training split's own most frequent first-listed genre). The fallback-row count it already prints per split is re-checked here independently, straight from `candidate_splits`' own `genres` column, rather than just trusting that prior print -- the two counts should agree exactly.

In [ ]:
genre_labels = eval_setup.derive_genre_label(candidate_splits)

print("genre_labels['test'].value_counts():")
print(genre_labels["test"].value_counts())
print()

for split in eval_setup.SPLITS:
    n_distinct = genre_labels[split].nunique()
    print(f"{split}: {n_distinct} distinct genres")
print()

# Independent re-check of the fallback count derive_genre_label
# already printed -- recomputed here directly from
# candidate_splits' own genres column (empty list, not NaN), not
# trusted from that prior cell's stdout.
for split in eval_setup.SPLITS:
    is_empty = candidate_splits[split][eval_setup.GENRE_COL].apply(len) == 0
    print(f"{split}: {is_empty.sum()} row(s) had an empty genres list (fallback expected)")

## 3. Shared XGBoost config and sensitive-attribute-safe features

`eval_setup.SharedXGBConfig` -- the one fixed, un-tuned config every representation is scored with (its own docstring covers why). Printed here so the exact hyperparameters feeding every later step are visible in this notebook's output, not just in source.

Also builds `raw_features_safe`/`classical_features_safe`: `eval_setup.drop_sensitive_attribute_columns` applied to `representations["raw"]`/`representations["classical"]` for every split -- [D46]'s guard against the recoverability probe trivially predicting `original_language` from a column that IS `original_language`.

In [ ]:
import dataclasses

xgb_config = eval_setup.SharedXGBConfig()

print("dataclasses.asdict(xgb_config):")
print(dataclasses.asdict(xgb_config))
print()
print("xgb_config.regressor_kwargs():")
print(xgb_config.regressor_kwargs())
print()
print("xgb_config.classifier_kwargs():")
print(xgb_config.classifier_kwargs())

raw_features_safe = {
    split: eval_setup.drop_sensitive_attribute_columns(representations["raw"][split], "raw")
    for split in eval_setup.SPLITS
}
classical_features_safe = {
    split: eval_setup.drop_sensitive_attribute_columns(
        representations["classical"][split], "classical"
    )
    for split in eval_setup.SPLITS
}

raw_before = representations["raw"]["test"].shape[1]
raw_after = raw_features_safe["test"].shape[1]
classical_before = representations["classical"]["test"].shape[1]
classical_after = classical_features_safe["test"].shape[1]

print()
print(f"raw (test split): {raw_before} columns before -> {raw_after} after (dropped {raw_before - raw_after})")
print(
    f"classical (test split): {classical_before} columns before -> "
    f"{classical_after} after (dropped {classical_before - classical_after})"
)